In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("MojaPierwszaAplikacja") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} - gotowy")

Spark 4.0.0-preview2 - gotowy


In [6]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [8]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [15]:
from pyspark.sql.functions import min as _min, max as _max

# TWÓJ KOD
# df.groupBy("category").agg(...).orderBy("category").show()
category_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
        _min("amount").alias("min_PLN"),
        _max("amount").alias("max_PLN")
    )
    .orderBy("category")
)

category_summary.show()

+-----------+---------+----------+-----------+-------+-------+
|   category|liczba_tx|  suma_PLN|srednia_PLN|min_PLN|max_PLN|
+-----------+---------+----------+-----------+-------+-------+
|elektronika|     2542|1520770.69|     598.26|    9.0| 9999.0|
|    książki|     2574| 851382.08|     330.76|    5.0|9107.25|
|     odzież|     2453| 849877.55|     346.46|    5.0|9696.63|
|    żywność|     2431| 789514.43|     324.77|    5.0|6916.92|
+-----------+---------+----------+-----------+-------+-------+



In [16]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [18]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [19]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [20]:
# Sortowanie od najstarszych (rosnąco) i wyświetlenie 100 wierszy
time_summary = df.orderBy("timestamp").limit(100)

time_summary.show(100, truncate=False)

+-------+-----------+--------+-------------------+-------+-------+
|amount |category   |store   |timestamp          |tx_id  |user_id|
+-------+-----------+--------+-------------------+-------+-------+
|111.9  |odzież     |Gdańsk  |2026-04-12 08:00:06|TX02146|u03    |
|567.73 |żywność    |Wrocław |2026-04-12 08:00:06|TX05535|u10    |
|58.01  |elektronika|Kraków  |2026-04-12 08:00:07|TX07268|u41    |
|5215.93|elektronika|Wrocław |2026-04-12 08:00:07|TX04640|u34    |
|99.43  |elektronika|Wrocław |2026-04-12 08:00:10|TX06112|u03    |
|34.49  |żywność    |Kraków  |2026-04-12 08:00:10|TX06172|u42    |
|719.28 |elektronika|Warszawa|2026-04-12 08:00:11|TX02607|u30    |
|41.87  |książki    |Warszawa|2026-04-12 08:00:16|TX05621|u23    |
|443.74 |elektronika|Gdańsk  |2026-04-12 08:00:20|TX05358|u01    |
|217.08 |odzież     |Wrocław |2026-04-12 08:00:21|TX03204|u39    |
|16.33  |żywność    |Gdańsk  |2026-04-12 08:00:21|TX06553|u49    |
|537.62 |elektronika|Kraków  |2026-04-12 08:00:23|TX06499|u50 

In [21]:
from pyspark.sql.functions import col

# Sortowanie od najnowszych (malejąco) i wyświetlenie 100 wierszy
last_100 = df.orderBy(col("timestamp").desc()).limit(100)

last_100.show(100, truncate=False)

+-------+-----------+--------+-------------------+-------+-------+
|amount |category   |store   |timestamp          |tx_id  |user_id|
+-------+-----------+--------+-------------------+-------+-------+
|350.17 |żywność    |Wrocław |2026-04-12 10:59:57|TX09329|u46    |
|1155.65|odzież     |Warszawa|2026-04-12 10:59:56|TX05571|u45    |
|25.46  |odzież     |Warszawa|2026-04-12 10:59:56|TX09059|u05    |
|747.59 |książki    |Wrocław |2026-04-12 10:59:49|TX04936|u09    |
|424.96 |książki    |Gdańsk  |2026-04-12 10:59:47|TX05643|u26    |
|158.7  |odzież     |Wrocław |2026-04-12 10:59:47|TX07519|u14    |
|104.17 |odzież     |Wrocław |2026-04-12 10:59:45|TX08603|u19    |
|22.31  |odzież     |Warszawa|2026-04-12 10:59:44|TX01843|u44    |
|144.71 |książki    |Wrocław |2026-04-12 10:59:43|TX02336|u49    |
|530.77 |żywność    |Warszawa|2026-04-12 10:59:40|TX04431|u44    |
|65.38  |odzież     |Gdańsk  |2026-04-12 10:59:36|TX03283|u27    |
|636.71 |elektronika|Gdańsk  |2026-04-12 10:59:30|TX00351|u50 